In [1]:
import json
import boto3
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
endpoint = os.getenv("S3_ENDPOINT", os.getenv("SEAWEEDFS_S3_ENDPOINT", "http://127.0.0.1:8333"))
if endpoint and not str(endpoint).startswith("http"):
    endpoint = f"http://{endpoint}"

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID", "admin"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY", "admin"),
    region_name=os.getenv("AWS_REGION", "us-east-1"),
)
print("S3 endpoint:", endpoint)

S3 endpoint: http://seaweedfs:8333


**JSON**

In [34]:
# Coger el JSON de esta "carpeta" y imprimirlo (si hay varios, usa el primero; cambia a keys[-1] si quieres el último)
bucket = "landing-zone"
prefix = "aemet/08019/2026/05/06/"  # misma ruta que listas arriba; acaba en /

keys = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        k = obj["Key"]
        if k.endswith(".json"):
            keys.append(k)

if not keys:
    raise FileNotFoundError(f"No hay ningún .json bajo s3://{bucket}/{prefix}")

key = keys[0]
print("Leyendo:", key, "\n")

raw = client.get_object(Bucket=bucket, Key=key)["Body"].read()
data = json.loads(raw.decode("utf-8"))
print(json.dumps(data, indent=2, ensure_ascii=False))

Leyendo: aemet/08019/2026/05/06/20_hourly_predictions.json 

{
  "id": "08019",
  "nombre": "Barcelona",
  "origen": {
    "web": "https://www.aemet.es",
    "enlace": "https://www.aemet.es/es/eltiempo/prediccion/municipios/horas/barcelona-id08019",
    "language": "es",
    "copyright": "© AEMET. Autorizado el uso de la información y su reproducción citando a AEMET como autora de la misma.",
    "notaLegal": "https://www.aemet.es/es/nota_legal",
    "productor": "Agencia Estatal de Meteorología - AEMET. Gobierno de España"
  },
  "version": "1.0",
  "elaborado": "2026-05-06T20:59:08",
  "provincia": "Barcelona",
  "prediccion": {
    "dia": [
      {
        "orto": "06:42",
        "fecha": "2026-05-06T00:00:00",
        "nieve": [
          {
            "value": "0",
            "periodo": "20"
          },
          {
            "value": "0",
            "periodo": "21"
          },
          {
            "value": "0",
            "periodo": "22"
          },
          {
       

**CSV**

In [17]:
import pandas as pd
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "landing-zone"
prefix = "ajuntament_bcn/accidents_gu/"

# Llista tots els CSVs
csvs = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".csv"):
            csvs.append(obj["Key"])

print(f"CSVs trobats: {len(csvs)}")

# Llegeix tots i ajunta'ls en un sol DataFrame
dfs = []
for key in csvs:
    raw = client.get_object(Bucket=bucket, Key=key)["Body"].read()
    df = pd.read_csv(io.BytesIO(raw))
    print(f"  → {key}: {len(df)} files -> columns: {df.columns} -> numero de nulos: {df.isna().sum()}")
    dfs.append(df)

# Ajunta tots
df_total = pd.concat(dfs, ignore_index=True)
print(f"\nTotal files combinades: {len(df_total)}")
print(df_total)


CSVs trobats: 5
  → ajuntament_bcn/accidents_gu/2021_ACCIDENTS_GU_BCN.csv: 7659 files -> columns: Index(['Numero_expedient', 'Codi_districte', 'Nom_districte', 'Codi_barri',
       'Nom_barri', 'Codi_carrer', 'Nom_carrer', 'Num_postal_caption',
       'Descripcio_dia_setmana', 'NK_Any', 'Mes_any', 'Nom_mes', 'Dia_mes',
       'Hora_dia', 'Descripcio_torn', 'Descripcio_causa_vianant',
       'Numero_morts', 'Numero_lesionats_lleus', 'Numero_lesionats_greus',
       'Numero_victimes', 'Numero_vehicles_implicats', 'Coordenada_UTM_X_ED50',
       'Coordenada_UTM_Y_ED50', 'Longitud_WGS84', 'Latitud_WGS84'],
      dtype='str') -> numero de nulos: Numero_expedient              0
Codi_districte                0
Nom_districte                 0
Codi_barri                    0
Nom_barri                     0
Codi_carrer                   0
Nom_carrer                    0
Num_postal_caption            0
Descripcio_dia_setmana        0
NK_Any                        0
Mes_any                       0

**XML**

In [ ]:
import boto3

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "landing-zone"
prefix = "sct//"

# Llista tots els XMLs
xmls = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".xml"):
            xmls.append(obj["Key"])

print(f"XMLs trobats: {len(xmls)}")

# Llegeix i imprimeix cada un
for key in xmls:
    print(f"\n{'='*50}")
    print(f"Fitxer: {key}")
    print('='*50)
    raw = client.get_object(Bucket=bucket, Key=key)["Body"].read()
    print(raw.decode("utf-8"))

XMLs trobats: 0


**PERSISTENT-LANDING-ZONE**

In [35]:
import pyarrow.parquet as pq
import pyarrow as pa
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "persistent-landing-zone"
prefix = "aemet/08019/2026/05/06/"

parquets = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            parquets.append(obj["Key"])

print(f"Parquets trobats: {len(parquets)}")

if parquets:
    raw = client.get_object(Bucket=bucket, Key=parquets[0])["Body"].read()
    
    # Llegeix ignorant els tipus d'extensió de pandas
    reader = pq.ParquetFile(io.BytesIO(raw))
    table = reader.read()
    
    # Converteix a pandas ignorant metadades problemàtiques
    df = table.to_pandas(timestamp_as_object=True)
    print(f"Total files: {len(df)}")
    print(df.columns)
    print(df.head())
    print(df.dtypes)
    

Parquets trobats: 1
Total files: 1
Index(['id', 'nombre', 'origen', 'version', 'elaborado', 'provincia',
       'prediccion'],
      dtype='str')
      id     nombre                                             origen  \
0  08019  Barcelona  {'web': 'https://www.aemet.es', 'enlace': 'htt...   

  version            elaborado  provincia  \
0     1.0  2026-05-06T20:59:08  Barcelona   

                                          prediccion  
0  {'dia': [{'orto': '06:42', 'fecha': '2026-05-0...  
id            str
nombre        str
origen        str
version       str
elaborado     str
provincia     str
prediccion    str
dtype: object


**TRUSTED ZONE**

In [36]:
import pyarrow.parquet as pq
import pyarrow as pa
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "trusted-zone"
prefix = "aemet/08019/2026/05/06/"

parquets = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            parquets.append(obj["Key"])

print(f"Parquets trobats: {len(parquets)}")

if parquets:
    raw = client.get_object(Bucket=bucket, Key=parquets[0])["Body"].read()
    
    # Llegeix ignorant els tipus d'extensió de pandas
    reader = pq.ParquetFile(io.BytesIO(raw))
    table = reader.read()
    
    # Converteix a pandas ignorant metadades problemàtiques
    df = table.to_pandas(timestamp_as_object=True)
    print(f"Total files: {len(df)}")
    print(df.columns)
    print(df.head())
    print(df.dtypes)

    

Parquets trobats: 1
Total files: 48
Index(['aemet_municipio_id', 'nombre_municipio', 'elaborado', 'provincia',
       'dia_fecha', 'orto', 'ocaso', 'hora_periodo', 'temperatura',
       'sens_termica', 'nieve', 'precipitacion', 'humedad_relativa',
       'prob_nieve', 'prob_precipitacion', 'prob_tormenta', 'estado_cielo',
       'estado_cielo_descripcion', 'viento_velocidad', 'viento_direccion',
       'viento_racha_max'],
      dtype='str')
  aemet_municipio_id nombre_municipio            elaborado  provincia  \
0              08019        Barcelona  2026-05-06T20:59:08  Barcelona   
1              08019        Barcelona  2026-05-06T20:59:08  Barcelona   
2              08019        Barcelona  2026-05-06T20:59:08  Barcelona   
3              08019        Barcelona  2026-05-06T20:59:08  Barcelona   
4              08019        Barcelona  2026-05-06T20:59:08  Barcelona   

             dia_fecha   orto  ocaso hora_periodo temperatura sens_termica  \
0  2026-05-06T00:00:00  06:42  20:54 